In [58]:
import nltk
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import DefaultDict, List, Tuple, Set, Hashable, Iterable, Dict
from collections import Counter, defaultdict
from tqdm import tqdm
from nltk import (
  CFG,  # Context-Free Grammar 
  PCFG  # Probabilistic Context-Free Grammar
)
from nltk.parse import (
  ChartParser,    # Chart Parsing Algorithm
  ViterbiParser   # Viterbi Parsing Algorithm
)
from nltk.tokenize import (
  word_tokenize,  # Tokenize a sentence into words
  sent_tokenize,  # Tokenize text into sentences
)
from spacy import displacy

try:
  nltk.data.find('tokenizers/punkt')
except LookupError:
  nltk.download('punkt')

nlp = spacy.load("es_core_news_sm")

In [59]:
sentences = [
  "Como fan de las series españolas y de Najwa, esto duele, la serie es muy mala",
  "Manu Ríos da para lo que da, enseñar cacho, Najwa hace de mala, papel repetido que no aporta ninguna capa nueva",
  "Telenovela de mediodía con un guión mediocre y diálogos planos",
  "En aspectos técnicos como fotografía, sonido, también deja que desear",
  "Lo peor de Carlos Montero, de largo."
]

# Tokenización
**TODO**: Integración del dataset en el notebook

In [60]:
tokens_by_sent = {}
for i,sent in enumerate(sentences):
  clean_sent = sent.lower()
  tokens_by_sent[i] = word_tokenize(clean_sent, language='spanish')

for idx in tokens_by_sent.keys():
  print(f"Doc: {idx+1}: {sentences[idx]}")
  print(f"Tokens: {tokens_by_sent[idx]}")

Doc: 1: Como fan de las series españolas y de Najwa, esto duele, la serie es muy mala
Tokens: ['como', 'fan', 'de', 'las', 'series', 'españolas', 'y', 'de', 'najwa', ',', 'esto', 'duele', ',', 'la', 'serie', 'es', 'muy', 'mala']
Doc: 2: Manu Ríos da para lo que da, enseñar cacho, Najwa hace de mala, papel repetido que no aporta ninguna capa nueva
Tokens: ['manu', 'ríos', 'da', 'para', 'lo', 'que', 'da', ',', 'enseñar', 'cacho', ',', 'najwa', 'hace', 'de', 'mala', ',', 'papel', 'repetido', 'que', 'no', 'aporta', 'ninguna', 'capa', 'nueva']
Doc: 3: Telenovela de mediodía con un guión mediocre y diálogos planos
Tokens: ['telenovela', 'de', 'mediodía', 'con', 'un', 'guión', 'mediocre', 'y', 'diálogos', 'planos']
Doc: 4: En aspectos técnicos como fotografía, sonido, también deja que desear
Tokens: ['en', 'aspectos', 'técnicos', 'como', 'fotografía', ',', 'sonido', ',', 'también', 'deja', 'que', 'desear']
Doc: 5: Lo peor de Carlos Montero, de largo.
Tokens: ['lo', 'peor', 'de', 'carlos', 'mo

# Procesar Textos con Spacy

In [61]:
def process_spacy(texts, batch_size=64):
  return nlp.pipe(texts, batch_size=batch_size)
docs = [doc for doc in tqdm(process_spacy(sentences, batch_size=64), total=len(sentences))]

100%|██████████| 5/5 [00:00<00:00, 255.67it/s]


**Leyenda**:
- `NOUN`: Sustantivo
- `VERB`: Verbo
- `ADJ`: Adjetivo
- `ADV`: Adverbio
- `PROPN`: Nombre Propio
- `DET`: Determinante
- `PRON`: Pronombre
- `ADP`: Preposición
- `CCONJ`: Conjunción
- `SCONJ`: Conjunción Sub
- `INTJ`: Interjección
- `NUM`: Número

In [62]:
categories = defaultdict(set)

tags = ["NOUN", "VERB", "AUX", "ADJ", "ADV", "PROPN", "DET", "PRON", "ADP", "CCONJ", "SCONJ", "INTJ", "NUM", "PUNCT"]
# Puede extenderse con: `NOUN__Gender=Masc|Number=Sing`, `NOUN__Gender=Fem|Number=Sing`, `VERB__Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin` 

for doc in docs:
  for token in doc:
    is_found = False
    for tag in tags:
      if token.pos_ == tag:
        categories[tag].add(token.text.lower())
        is_found = True
    if not is_found:
      print(f"Dont Found: {token.text} | {token.pos_}")
      # categories["OTHER"].add(token.text.lower())

for idx in categories.keys():
  print(f"TAG({idx}) = {categories[idx]}")

TAG(SCONJ) = {'como', 'que'}
TAG(NOUN) = {'diálogos', 'sonido', 'capa', 'guión', 'largo', 'mediodía', 'cacho', 'papel', 'serie', 'mala', 'series', 'aspectos', 'fan', 'fotografía', 'repetido'}
TAG(ADP) = {'de', 'en', 'para', 'con'}
TAG(DET) = {'la', 'un', 'las', 'ninguna'}
TAG(ADJ) = {'nueva', 'planos', 'españolas', 'técnicos', 'mala', 'mediocre', 'peor'}
TAG(CCONJ) = {'y'}
TAG(PROPN) = {'carlos', 'ríos', 'najwa', 'telenovela', 'montero', 'manu'}
TAG(PUNCT) = {',', '.'}
TAG(PRON) = {'que', 'esto', 'lo'}
TAG(VERB) = {'enseñar', 'da', 'desear', 'hace', 'deja', 'duele', 'aporta'}
TAG(AUX) = {'es'}
TAG(ADV) = {'muy', 'también', 'no'}


In [63]:
from typing import Hashable


def invert_defaultdict_of_sets(d: DefaultDict[Hashable, Set[Hashable]]) -> DefaultDict[Hashable, Set[Hashable]]:
  inv: DefaultDict[Hashable, Set[Hashable]] = defaultdict(set)
  for k, vs in d.items():
    for v in vs:
      inv[v].add(k)
  return inv

def invert_defaultdict_of_lists(d: DefaultDict[Hashable, List[Hashable]]) -> DefaultDict[Hashable, List[Hashable]]:
  inv: DefaultDict[Hashable, List[Hashable]] = defaultdict(list)
  for k, vs in d.items():
    for v in vs:
      inv[v].append(k)
  return inv

def invert_dict(d: Dict[Hashable, Hashable]) -> Dict[Hashable, Hashable]:
  return {v: k for k, v in d.items()}

def invert_dict_multi(d: Dict[Hashable, Iterable[Hashable]]) -> DefaultDict[Hashable, Set[Hashable]]:
  inv: DefaultDict[Hashable, Set[Hashable]] = defaultdict(set)
  for k, vs in d.items():
    for v in vs:
      inv[v].add(k)
  return inv

inverse_categories = invert_defaultdict_of_sets(categories)
for idx in inverse_categories.keys():
  inverse_categories[idx] = list[Hashable](inverse_categories[idx])[0]
  print(f"WORD({idx}) = TAG({ inverse_categories[idx] })")

WORD(como) = TAG(SCONJ)
WORD(que) = TAG(SCONJ)
WORD(diálogos) = TAG(NOUN)
WORD(sonido) = TAG(NOUN)
WORD(capa) = TAG(NOUN)
WORD(guión) = TAG(NOUN)
WORD(largo) = TAG(NOUN)
WORD(mediodía) = TAG(NOUN)
WORD(cacho) = TAG(NOUN)
WORD(papel) = TAG(NOUN)
WORD(serie) = TAG(NOUN)
WORD(mala) = TAG(ADJ)
WORD(series) = TAG(NOUN)
WORD(aspectos) = TAG(NOUN)
WORD(fan) = TAG(NOUN)
WORD(fotografía) = TAG(NOUN)
WORD(repetido) = TAG(NOUN)
WORD(de) = TAG(ADP)
WORD(en) = TAG(ADP)
WORD(para) = TAG(ADP)
WORD(con) = TAG(ADP)
WORD(la) = TAG(DET)
WORD(un) = TAG(DET)
WORD(las) = TAG(DET)
WORD(ninguna) = TAG(DET)
WORD(nueva) = TAG(ADJ)
WORD(planos) = TAG(ADJ)
WORD(españolas) = TAG(ADJ)
WORD(técnicos) = TAG(ADJ)
WORD(mediocre) = TAG(ADJ)
WORD(peor) = TAG(ADJ)
WORD(y) = TAG(CCONJ)
WORD(carlos) = TAG(PROPN)
WORD(ríos) = TAG(PROPN)
WORD(najwa) = TAG(PROPN)
WORD(telenovela) = TAG(PROPN)
WORD(montero) = TAG(PROPN)
WORD(manu) = TAG(PROPN)
WORD(,) = TAG(PUNCT)
WORD(.) = TAG(PUNCT)
WORD(esto) = TAG(PRON)
WORD(lo) = TAG(PRON)

In [64]:
for idx in tokens_by_sent.keys():
  print(f"Doc: {idx+1}: {sentences[idx]}")
  print([ inverse_categories[token] for token in tokens_by_sent[idx] ])

Doc: 1: Como fan de las series españolas y de Najwa, esto duele, la serie es muy mala
['SCONJ', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADJ', 'CCONJ', 'ADP', 'PROPN', 'PUNCT', 'PRON', 'VERB', 'PUNCT', 'DET', 'NOUN', 'AUX', 'ADV', 'ADJ']
Doc: 2: Manu Ríos da para lo que da, enseñar cacho, Najwa hace de mala, papel repetido que no aporta ninguna capa nueva
['PROPN', 'PROPN', 'VERB', 'ADP', 'PRON', 'SCONJ', 'VERB', 'PUNCT', 'VERB', 'NOUN', 'PUNCT', 'PROPN', 'VERB', 'ADP', 'ADJ', 'PUNCT', 'NOUN', 'NOUN', 'SCONJ', 'ADV', 'VERB', 'DET', 'NOUN', 'ADJ']
Doc: 3: Telenovela de mediodía con un guión mediocre y diálogos planos
['PROPN', 'ADP', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADJ', 'CCONJ', 'NOUN', 'ADJ']
Doc: 4: En aspectos técnicos como fotografía, sonido, también deja que desear
['ADP', 'NOUN', 'ADJ', 'SCONJ', 'NOUN', 'PUNCT', 'NOUN', 'PUNCT', 'ADV', 'VERB', 'SCONJ', 'VERB']
Doc: 5: Lo peor de Carlos Montero, de largo.
['PRON', 'ADJ', 'ADP', 'PROPN', 'PROPN', 'PUNCT', 'ADP', 'NOUN', 'PUNCT']


# Construcción de Reglas CFG

In [84]:
S1 = """ 
S -> PRELUDE PUNCT CLAUSE PUNCT CLAUSE
PRELUDE -> SCONJ NP
CLAUSE -> NP VP
NP -> NP ADP NP CCONJ ADP NP
NP -> DET NOUN | DET NOUN ADJ
NP -> NOUN | PRON | PROPN 
VP -> VERB
VP -> AUX ADV ADJ
"""

S2 = """ 
S2 -> CLAUSE PUNCT CLAUSE PUNCT CLAUSE PUNCT CLAUSE

CLAUSE -> NP VP | VP NP
CLAUSE -> NP SCONJ VP

NP -> PROPN | PROPN NP
NP -> NOUN | NOUN NP   
NP -> DET NOUN ADJ

VP -> VERB ADP PRON SCONJ VERB
VP -> VERB | VERB ADP ADJ
VP -> ADV VERB NP
"""

S3 = """ 
S3 -> NP
NP -> PROPN | PROPN PP | PROPN NP PUNCT NP | PROPN NP CCONJ NP
NP -> PP
NP -> DET NP 
NP -> NOUN | NOUN ADJ 
PP -> ADP NP | ADP NP PP
"""

S4 = """ 
S4 -> CLAUSE SCONJ NP CLAUSE
CLAUSE -> ADP NP | ADV VERB SCONJ VERB
NP -> NOUN ADJ
NP -> NOUN PUNCT | NOUN PUNCT NP | NOUN
"""

S5 = """ 
S -> NP PP PUNCT PP PUNCT
NP -> PRON ADJ
PP -> ADP NPP | ADP NOUN 
NPP -> PROPN PROPN
PUNCT -> COMMA
"""

In [85]:
def build_cfg(categories, ind_grammar, head_grammar):
  lexical_rules = []
  for tag, items in categories.items():
    if items:
      alts = " | ".join(sorted({f"'{w}'" for w in items}))
      lexical_rules.append(f"{tag} -> {alts}")
  
  return "\n".join(head_grammar + ind_grammar + lexical_rules)


head_grammar = ["S -> " + " | ".join([f"S{i+1}" for i in range(len(sentences))])]
ind_grammar = [S1, S2, S3, S4, S5]

grammar_text = build_cfg(categories, ind_grammar, head_grammar)
grammar = CFG.fromstring(grammar_text)
print(grammar)

Grammar with 108 productions (start state = S)
    S -> S1
    S -> S2
    S -> S3
    S -> S4
    S -> S5
    S -> PRELUDE PUNCT CLAUSE PUNCT CLAUSE
    PRELUDE -> SCONJ NP
    CLAUSE -> NP VP
    NP -> NP ADP NP CCONJ ADP NP
    NP -> DET NOUN
    NP -> DET NOUN ADJ
    NP -> NOUN
    NP -> PRON
    NP -> PROPN
    VP -> VERB
    VP -> AUX ADV ADJ
    S2 -> CLAUSE PUNCT CLAUSE PUNCT CLAUSE PUNCT CLAUSE
    CLAUSE -> NP VP
    CLAUSE -> VP NP
    CLAUSE -> NP SCONJ VP
    NP -> PROPN
    NP -> PROPN NP
    NP -> NOUN
    NP -> NOUN NP
    NP -> DET NOUN ADJ
    VP -> VERB ADP PRON SCONJ VERB
    VP -> VERB
    VP -> VERB ADP ADJ
    VP -> ADV VERB NP
    S3 -> NP
    NP -> PROPN
    NP -> PROPN PP
    NP -> PROPN NP PUNCT NP
    NP -> PROPN NP CCONJ NP
    NP -> PP
    NP -> DET NP
    NP -> NOUN
    NP -> NOUN ADJ
    PP -> ADP NP
    PP -> ADP NP PP
    S4 -> CLAUSE SCONJ NP CLAUSE
    CLAUSE -> ADP NP
    CLAUSE -> ADV VERB SCONJ VERB
    NP -> NOUN ADJ
    NP -> NOUN PUNCT
    NP 

In [86]:
parser = ChartParser(grammar)
parsed = []
for i in range(len(sentences)):
  toks = tokens_by_sent[i]
  trees = list(parser.parse(toks))
  print(f"Sentence {i+1} parses: {len(trees)}")
  parsed.append(trees)

Sentence 1 parses: 4
Sentence 2 parses: 4
Sentence 3 parses: 4
Sentence 4 parses: 1
Sentence 5 parses: 4
